In [14]:
!pip install requests;

!pip install panda;


import os
import requests
import pandas as pd

# ==========================================
# CONFIGURATION / CREDENTIALS
# ==========================================
STEAM_API_KEY = "E56D21B82A054BEAB30A2C62A538EE1A"
STEAM_ID = "76561199483779798"

NOTION_TOKEN = "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"
NOTION_DATABASE_ID = "YOUR_NOTION_DATABASE_ID"

# ==========================================
# 1. Get_owned_game (Steam API to DataFrame)
# ==========================================
def get_owned_games(api_key: str, steam_id: str) -> pd.DataFrame:
    """
    Fetches owned games for a Steam user and returns a Pandas DataFrame.
    Equivalent to the R 'Get_owned_game' utility.
    """
    url = "http://api.steampowered.com/IPlayerService/GetOwnedGames/v0001/"
    params = {
        "key": api_key,
        "steamid": steam_id,
        "include_appinfo": True,       # Includes game names and icon URLs
        "include_played_free_games": True,
        "format": "json"
    }
    
    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()
        
        games_list = data.get("response", {}).get("games", [])
        
        if not games_list:
            print("No games found or profile is private.")
            return pd.DataFrame()
        
        # Convert JSON list directly to Pandas DataFrame
        df = pd.DataFrame(games_list)
        
        # Clean up columns: Convert playtime from minutes to hours if desired
        if "playtime_forever" in df.columns:
            df["playtime_hours"] = (df["playtime_forever"] / 60).round(2)
            
        return df
        
    except requests.exceptions.RequestException as e:
        print(f"Error fetching data from Steam API: {e}")
        return pd.DataFrame()


# ==========================================
# 2. df2Notion Utils (DataFrame to Notion API)
# ==========================================
def create_notion_page(database_id: str, headers: dict, game_data: dict):
    """
    Helper function to insert a single row (game) into a Notion Database.
    """
    url = "https://api.notion.com/v1/pages"
    
    # Extract data with fallbacks if keys don't exist
    app_id = str(game_data.get("appid", ""))
    game_name = game_data.get("name", "Unknown Game")
    playtime_hours = game_data.get("playtime_hours", 0.0)
    
    # Construct the Notion properties payload based on your database schema
    payload = {
        "parent": {"database_id": database_id},
        "properties": {
            "Game Name": {
                "title": [
                    {
                        "text": {
                            "content": game_name
                        }
                    }
                ]
            },
            "AppID": {
                "rich_text": [
                    {
                        "text": {
                            "content": app_id
                        }
                    }
                ]
            },
            "Playtime (Hours)": {
                "number": playtime_hours
            }
        }
    }
    
    response = requests.post(url, json=payload, headers=headers)
    if response.status_code != 200:
        print(f"Failed to insert {game_name}: {response.text}")


def df_to_notion(df: pd.DataFrame, token: str, database_id: str):
    """
    Iterates through the DataFrame and populates the Notion Database.
    Equivalent to 'df2Notion' in R.
    """
    if df.empty:
        print("DataFrame is empty. Skipping Notion update.")
        return

    # Notion API Headers
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
        "Notion-Version": "2022-06-28"  # Use the modern stable Notion API version
    }
    
    print(f"Starting export of {len(df)} games to Notion...")
    
    # Convert DataFrame rows to dictionaries and push to Notion
    for _, row in df.iterrows():
        game_data = row.to_dict()
        create_notion_page(database_id, headers, game_data)
        
    print("Notion update completed successfully!")


# ==========================================
# MAIN EXECUTION
# ==========================================
if __name__ == "__main__":
    # 1. Fetch data from Steam
    df_games = get_owned_games(STEAM_API_KEY, STEAM_ID)
    
    if not df_games.empty:
        # Optional: Inspect your dataframe local structure
        print(df_games.head())



[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
   appid                          name  playtime_forever  \
0    220                   Half-Life 2                 0   
1    320       Half-Life 2: Deathmatch                 0   
2    360  Half-Life Deathmatch: Source                 0   
3    730              Counter-Strike 2                10   
4   6100                          Eets                 0   

                               img_icon_url has_community_visible_stats  \
0  fcfb366051782b8ebf2aa297f3b746395858cb62                        True   
1  795e85364189511f4990861b578084deef086cb1                         NaN   
2  40b8a62efff5a9ab356e5c56f5c8b0532c8e1aa3                         NaN   
3  8dbc71957312bbd3baea65848b545be9eae2a355                        True   
4  fbaf69a669637c8bd5cebdd9a8f06b4f378

In [15]:
import pandas as pd
import requests

def load_steam_ids(file_path: str = "steamid.txt") -> list:
    """Reads Steam IDs from a text file, ignoring empty lines and comments."""
    if not os.path.exists(file_path):
        print(
            f"Warning: '{file_path}' not found. Returning an empty account list."
        )
        return []

    steam_ids = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            clean_line = line.strip()

            # Ignore empty lines or comment lines starting with '#'
            if not clean_line or clean_line.startswith("#"):
                continue

            steam_ids.append(clean_line)

    return steam_ids


def get_owned_games(api_key: str, steam_id: str) -> pd.DataFrame:
    """Fetches owned games for a single Steam account and attaches the Steam ID to each row."""
    url = "http://api.steampowered.com/IPlayerService/GetOwnedGames/v0001/"
    params = {
        "key": api_key,
        "steamid": steam_id,
        "include_appinfo": True,
        "include_free_sub": True,
        "include_played_free_games": True,
        "format": "json",
        "skip_unvetted_apps": False,
        "include_extended_appinfo": True,
    }

    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()

        games_list = data.get("response", {}).get("games", [])

        if not games_list:
            print(f"No games found or profile is private for Steam ID: {steam_id}")
            return pd.DataFrame()

        df = pd.DataFrame(games_list)

        # Keep only the columns we need
        ##columns_to_keep = ["appid", "name", "playtime_forever"]
        ##df = df[[col for col in columns_to_keep if col in df.columns]].copy()
        

        # Add the owner's Steam ID as a string column so we can track it
        df["steam_account_id"] = str(steam_id)

        return df

    except requests.exceptions.RequestException as e:
        print(f"Error fetching data for Steam ID {steam_id}: {e}")
        return pd.DataFrame()


def merge_steam_accounts(
    api_key: str, steam_ids: list, output_csv: str = "merged_steam_games.csv"
):
    """Fetches games from multiple accounts, merges them by appid, sums playtime,

    and lists all owning Steam IDs comma-separated.
    """
    all_account_dfs = []

    for idx, steam_id in enumerate(steam_ids, start=1):
        print(f"Fetching data for Account {idx} ({steam_id})...")
        df = get_owned_games(api_key, steam_id)

        if not df.empty:
            all_account_dfs.append(df)

    if not all_account_dfs:
        print("No data collected from any of the accounts. CSV not created.")
        return

    # 1. Combine all individual dataframes on top of each other
    combined_df = pd.concat(all_account_dfs, ignore_index=True)

    # 2. Merge rows with the same appid using named aggregation
    # - playtime_forever: sum the minutes
    # - steam_account_ids: join unique IDs with a comma
    merged_df = (
        combined_df.groupby(["appid", "name"], as_index=False)
        .agg(
            playtime_forever=("playtime_forever", "sum"),
            steam_account_ids=(
                "steam_account_id",
                lambda x: ", ".join(x.unique()),
            ),
        )
    )

    # Convert playtime from minutes to hours
    merged_df["playtime_hours"] = (merged_df["playtime_forever"] / 60).round(2)

    # Sort alphabetically by game name
    merged_df = merged_df.sort_values(by="name").reset_index(drop=True)

    # Reorder columns for better readability in CSV
    final_columns = [
        "appid",
        "name",
        "playtime_forever",
        "playtime_hours",
        "steam_account_ids",
    ]
    ##merged_df = merged_df[final_columns]

    # Export to CSV
    merged_df.to_csv(output_csv, index=False)
    print(f"\n Success! Merged data saved to: {output_csv}")
    print(f"Total unique games found: {len(merged_df)}")


# ==========================================
# CONFIGURATION & EXECUTION
# ==========================================
if __name__ == "__main__":

    STEAM_API_KEY = "E56D21B82A054BEAB30A2C62A538EE1A"


# Load accounts dynamically from the txt file
    STEAM_ACCOUNTS = load_steam_ids("steamid.txt")

    print(f"Loaded {len(STEAM_ACCOUNTS)} Steam account(s) from file.")
    # Add as many Steam 64 IDs as you want to merge here
    STEAM_ACCOUNTS = [
        "76561199483779798",
        "76561198116491768",
        "76561198120249474",
    ]

    OUTPUT_FILE = "what_merged_steam_library.csv"

    merge_steam_accounts(
        api_key=STEAM_API_KEY, steam_ids=STEAM_ACCOUNTS, output_csv=OUTPUT_FILE
    )

Loaded 3 Steam account(s) from file.
Fetching data for Account 1 (76561199483779798)...
Fetching data for Account 2 (76561198116491768)...
Fetching data for Account 3 (76561198120249474)...

 Success! Merged data saved to: what_merged_steam_library.csv
Total unique games found: 127


In [16]:
import os
import pandas as pd
import requests


def load_steam_ids(file_path: str = "steamid.txt") -> list:
    """Reads Steam IDs from a text file, ignoring empty lines and comments."""
    if not os.path.exists(file_path):
        print(
            f"Warning: '{file_path}' not found. Returning an empty account list."
        )
        return []

    steam_ids = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            clean_line = line.strip()
            if not clean_line or clean_line.startswith("#"):
                continue
            steam_ids.append(clean_line)
    return steam_ids


def get_owned_games(api_key: str, steam_id: str) -> pd.DataFrame:
    """Fetches owned games with extended appinfo and maps the owner's ID."""
    url = "http://api.steampowered.com/IPlayerService/GetOwnedGames/v0001/"
    params = {
        "key": api_key,
        "steamid": steam_id,
        "include_appinfo": True,
        "include_extended_appinfo": True,  # Pulls advanced metadata flags
        "include_played_free_games": True,
        "format": "json",
    }

    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()

        games_list = data.get("response", {}).get("games", [])

        if not games_list:
            print(f"No games found or profile is private for Steam ID: {steam_id}")
            return pd.DataFrame()

        # Load everything directly into a DataFrame to capture all dynamic columns
        df = pd.DataFrame(games_list)

        # Track ownership mapping
        df["steam_account_id"] = str(steam_id)

        return df

    except requests.exceptions.RequestException as e:
        print(f"Error fetching data for Steam ID {steam_id}: {e}")
        return pd.DataFrame()


def merge_steam_accounts(
    api_key: str, steam_ids: list, output_csv: str = "merged_steam_games.csv"
):
    """Fetches games from multiple accounts, dynamically aggregates all available columns

    by appid, and outputs a complete dataset.
    """
    all_account_dfs = []

    for idx, steam_id in enumerate(steam_ids, start=1):
        print(f"Fetching data for Account {idx} ({steam_id})...")
        df = get_owned_games(api_key, steam_id)

        if not df.empty:
            all_account_dfs.append(df)

    if not all_account_dfs:
        print("No data collected from any of the accounts. CSV not created.")
        return

    # Combine all individual dataframes flat
    combined_df = pd.concat(all_account_dfs, ignore_index=True)

    # Automatically identify numerical playtime columns vs descriptive metadata columns
    playtime_cols = [col for col in combined_df.columns if "playtime" in col]
    metadata_cols = [
        col
        for col in combined_df.columns
        if col not in playtime_cols
        and col not in ["appid", "steam_account_id"]
    ]

    # Build an aggregation dictionary dynamically for every column present
    agg_rules = {}

    # 1. Sum up all tracking metrics (forever, 2weeks, deck, windows, mac, linux, etc.)
    for col in playtime_cols:
        agg_rules[col] = "sum"

    # 2. Concat the list of account IDs ownership strings
    agg_rules["steam_account_id"] = lambda x: ", ".join(x.dropna().astype(str).unique())

    # 3. For any other metadata columns from extended info, take the first non-null value
    for col in metadata_cols:
        agg_rules[col] = "first"

    # Group and merge on unique App ID
    merged_df = combined_df.groupby("appid", as_index=False).agg(agg_rules)

    # Optional: Calculate clean hours safely for 'playtime_forever' if it's there
    if "playtime_forever" in merged_df.columns:
        merged_df["playtime_hours"] = (merged_df["playtime_forever"] / 60).round(2)

    # Clean sorting (Fallback to appid if 'name' string column failed to map)
    sort_by = "name" if "name" in merged_df.columns else "appid"
    merged_df = merged_df.sort_values(by=sort_by).reset_index(drop=True)

    # Rename tracking column cleanly
    merged_df = merged_df.rename(columns={"steam_account_id": "steam_account_ids"})

    # Save to file
    merged_df.to_csv(output_csv, index=False)
    print(f"\n Success! Fully detailed data saved to: {output_csv}")
    print(f"Total unique games found: {len(merged_df)}")
    print(f"Columns exported: {list(merged_df.columns)}")


# ==========================================
# CONFIGURATION & EXECUTION
# ==========================================
if __name__ == "__main__":
    STEAM_API_KEY = "E56D21B82A054BEAB30A2C62A538EE1A"

    OUTPUT_FILE = "merged_steam_extended_library.csv"

    # Load account values from local file path
    STEAM_ACCOUNTS = load_steam_ids("steamid.txt")

    print(f"Loaded {len(STEAM_ACCOUNTS)} Steam account(s) from file.")

    if STEAM_ACCOUNTS:
        merge_steam_accounts(
            api_key=STEAM_API_KEY, steam_ids=STEAM_ACCOUNTS, output_csv=OUTPUT_FILE
        )

Loaded 3 Steam account(s) from file.
Fetching data for Account 1 (76561199483779798)...


Fetching data for Account 2 (76561198116491768)...
Fetching data for Account 3 (76561198120249474)...

 Success! Fully detailed data saved to: merged_steam_extended_library.csv
Total unique games found: 126
Columns exported: ['appid', 'playtime_forever', 'playtime_2weeks', 'steam_account_ids', 'name', 'img_icon_url', 'has_community_visible_stats', 'capsule_filename', 'has_workshop', 'has_market', 'has_dlc', 'content_descriptorids', 'has_leaderboards', 'sort_as', 'playtime_hours']
